In [1]:
# =========================
# Cell 0 — Determinism + /data-first outputs
# =========================
import os, json, random, re, time
from glob import glob
from collections import Counter
from datetime import datetime

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import SimpleITK as sitk
from PIL import Image

SEED = 42

def set_determinism(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

set_determinism(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

PREPROC_ROOT = "/data/Preprocessed"
assert os.path.isdir(PREPROC_ROOT), f"Missing: {PREPROC_ROOT}"

TARGET_HW = 512  # MedSAM2 t512

# /data-first outputs
RUN_NAME = datetime.now().strftime("medsam2_refine_%Y%m%d_%H%M%S")
OUT_ROOT = f"/data/jrbonill/medsam2_refine_runs/{RUN_NAME}"
TB_DIR   = f"{OUT_ROOT}/tb"
CKPT_OUT = f"{OUT_ROOT}/checkpoints"
TMP_DIR  = f"{OUT_ROOT}/tmp"

os.makedirs(TB_DIR, exist_ok=True)
os.makedirs(CKPT_OUT, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

os.environ["TMPDIR"] = TMP_DIR
os.environ["XDG_CACHE_HOME"] = f"{OUT_ROOT}/cache"
os.makedirs(os.environ["XDG_CACHE_HOME"], exist_ok=True)

print("OUT_ROOT:", OUT_ROOT)
print("TB_DIR  :", TB_DIR)
print("CKPT_OUT:", CKPT_OUT)

DEVICE: cuda
OUT_ROOT: /data/jrbonill/medsam2_refine_runs/medsam2_refine_20260128_173900
TB_DIR  : /data/jrbonill/medsam2_refine_runs/medsam2_refine_20260128_173900/tb
CKPT_OUT: /data/jrbonill/medsam2_refine_runs/medsam2_refine_20260128_173900/checkpoints


In [2]:
# =========================
# Cell 1 — Pairing logic (UNCHANGED from your MedSAM2-Refine)
# =========================
_idx_pattern      = re.compile(r"_(?:img|frame|slice)(\d+)")
_mask_idx_pattern = re.compile(r"_mask(\d+)")

def _normalize_modality(mod):
    if mod is None:
        return None
    return str(mod).strip().lower()

def _stem_key(name: str) -> str:
    nm = name.lower()
    stem = os.path.splitext(nm)[0]
    stem = re.sub(r'(?:_mask|-mask|\.mask|_gt|_label|-label|_anno|_annotation)', '', stem)
    stem = re.sub(r'[^a-z0-9]+', '_', stem).strip('_')
    return stem

def _ext(path):
    p = path.lower()
    if p.endswith(".nii.gz"):
        return ".nii.gz"
    return os.path.splitext(p)[1].lower()

IMG_EXT_2D = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def _dim_from_image_path(p):
    return 2 if _ext(p) in IMG_EXT_2D else 3

def _pairs_from_entry(ds_name, sub_name, split, entry, sub_modality=None, sub_pipeline=None):
    """
    entry expects:
      - proc_images: list[str]
      - proc_masks: list[{"path": str, "class": str}]
    Returns flat list of pair dicts.
    """
    imgs = entry.get("proc_images", []) or []
    msks = entry.get("proc_masks", []) or []
    if not imgs or not msks:
        return []

    entry_modality = _normalize_modality(entry.get("subdataset_modality", sub_modality))
    entry_pipeline = entry.get("subdataset_pipeline", sub_pipeline)

    # index -> image map from _(img|frame|slice)####
    idx_map = {}
    for ip in imgs:
        base = os.path.basename(ip)
        m = _idx_pattern.search(base)
        if m:
            idx_map[int(m.group(1))] = ip

    # stem key -> image map
    img_by_key = {}
    for ip in imgs:
        img_by_key[_stem_key(os.path.basename(ip))] = ip

    out = []

    for mrec in msks:
        msk_path = mrec.get("path", None)
        cls_name = mrec.get("class", None)
        if not msk_path or cls_name is None:
            continue

        base = os.path.basename(msk_path)

        # 1) prefer _mask#### (PAIP2019 style)
        m = _mask_idx_pattern.search(base)
        if m:
            idx = int(m.group(1))
            ip = idx_map.get(idx)
            if ip:
                out.append({
                    "dataset": ds_name,
                    "subdataset": sub_name,
                    "split": split,
                    "modality": entry_modality or "unknown",
                    "pipeline": (entry_pipeline or ""),
                    "identifier": entry.get("identifier", None),
                    "image": ip,
                    "mask": msk_path,
                    "class": cls_name,
                    "dim": _dim_from_image_path(ip),
                })
                continue

        # 2) fallback _(img|frame|slice)####
        m = _idx_pattern.search(base)
        if m:
            idx = int(m.group(1))
            ip = idx_map.get(idx)
            if ip:
                out.append({
                    "dataset": ds_name,
                    "subdataset": sub_name,
                    "split": split,
                    "modality": entry_modality or "unknown",
                    "pipeline": (entry_pipeline or ""),
                    "identifier": entry.get("identifier", None),
                    "image": ip,
                    "mask": msk_path,
                    "class": cls_name,
                    "dim": _dim_from_image_path(ip),
                })
                continue

        # 3) last resort: normalized stem match
        key = _stem_key(base)
        ip = img_by_key.get(key)
        if ip:
            out.append({
                "dataset": ds_name,
                "subdataset": sub_name,
                "split": split,
                "modality": entry_modality or "unknown",
                "pipeline": (entry_pipeline or ""),
                "identifier": entry.get("identifier", None),
                "image": ip,
                "mask": msk_path,
                "class": cls_name,
                "dim": _dim_from_image_path(ip),
            })

    return out

def load_all_pairs_from_preprocessed(preproc_root, splits=("train","test")):
    ds_names = sorted([d for d in os.listdir(preproc_root) if os.path.isdir(os.path.join(preproc_root, d))])

    group_files = []
    for ds in ds_names:
        gf = os.path.join(preproc_root, ds, f"{ds}_groups.json")
        if os.path.isfile(gf):
            group_files.append(gf)

    assert group_files, f"No <ds>/<ds>_groups.json found under {preproc_root}"

    print("datasets:", len(ds_names))
    print("group files:", len(group_files))
    print("sample:", group_files[0])

    pairs = []
    bad = 0

    for gf in group_files:
        ds_name = os.path.basename(gf).replace("_groups.json","")
        try:
            data = json.load(open(gf, "r"))
        except Exception:
            bad += 1
            continue

        for sub in data.get("subdatasets", []):
            sub_name     = sub.get("name", "default")
            sub_modality = _normalize_modality(sub.get("modality", None))
            sub_pipeline = sub.get("pipeline", sub.get("subdataset_pipeline", None))

            for split in splits:
                for entry in sub.get(split, []):
                    if entry.get("split", split) != split:
                        continue
                    pairs.extend(_pairs_from_entry(
                        ds_name=ds_name,
                        sub_name=sub_name,
                        split=split,
                        entry=entry,
                        sub_modality=sub_modality,
                        sub_pipeline=sub_pipeline,
                    ))

    return pairs, bad

pairs, bad = load_all_pairs_from_preprocessed(PREPROC_ROOT, splits=("train","test"))
print("total paired samples:", len(pairs), "bad group files:", bad)

print("Splits:", Counter(p["split"] for p in pairs))
print("Dims:", Counter(p["dim"] for p in pairs))
print("Top modalities:", Counter(p["modality"] for p in pairs).most_common(10))
print("Sample:", pairs[0] if pairs else None)

if pairs:
    p0 = pairs[0]
    print("exists image:", os.path.exists(p0["image"]), "exists mask:", os.path.exists(p0["mask"]))

datasets: 57
group files: 57
sample: /data/Preprocessed/2018-data-science-bowl/2018-data-science-bowl_groups.json
total paired samples: 743586 bad group files: 0
Splits: Counter({'train': 639907, 'test': 103679})
Dims: Counter({2: 556722, 3: 186864})
Top modalities: [('histopathology', 310620), ('x-ray', 194435), ('mri', 123497), ('ct', 70173), ('ultrasound', 22649), ('fetoscopy', 8002), ('retinal-scan', 6544), ('colonoscopy', 3941), ('dermoscopy', 3725)]
Sample: {'dataset': '2018-data-science-bowl', 'subdataset': 'default', 'split': 'train', 'modality': 'histopathology', 'pipeline': '2D', 'identifier': 'train_003cee89357d9fe13516167fd67b609a164651b21934585648c740d2c3d86dc1', 'image': '/data/Preprocessed/2018-data-science-bowl/histopathology/default/train/003cee89357d9fe13516167fd67b609a164651b21934585648c740d2c3d86dc1/images/b4f0a877_img000.png', 'mask': '/data/Preprocessed/2018-data-science-bowl/histopathology/default/train/003cee89357d9fe13516167fd67b609a164651b21934585648c740d2c3d8

In [3]:
# =========================
# Cell 2 — Deterministic splits + optional deterministic subsampling
# =========================
def make_splits(pairs, val_frac=0.02, seed=42):
    train = [p for p in pairs if p["split"] == "train"]
    test  = [p for p in pairs if p["split"] == "test"]

    rng = random.Random(seed)
    rng.shuffle(train)

    n_val = int(len(train) * val_frac)
    val = train[:n_val]
    train = train[n_val:]
    return train, val, test

train_pairs, val_pairs, test_pairs = make_splits(pairs, val_frac=0.02, seed=SEED)
print("train/val/test:", len(train_pairs), len(val_pairs), len(test_pairs))

MAX_TRAIN_SAMPLES = None  # optional
MAX_VAL_SAMPLES   = None  # optional

def deterministic_subset(records, max_n, seed):
    if (max_n is None) or (len(records) <= max_n):
        return records
    rng = np.random.RandomState(seed)
    idx = np.arange(len(records))
    rng.shuffle(idx)
    idx = idx[:max_n]
    return [records[i] for i in idx]

train_pairs_small = deterministic_subset(train_pairs, MAX_TRAIN_SAMPLES, SEED)
val_pairs_small   = deterministic_subset(val_pairs,   MAX_VAL_SAMPLES,   SEED + 1)

print("train_pairs_small:", len(train_pairs_small))
print("val_pairs_small  :", len(val_pairs_small))

train/val/test: 627109 12798 103679
train_pairs_small: 627109
val_pairs_small  : 12798


In [4]:
# =========================
# Cell 3 — ImageNet normalization utilities (KEEP for MedSAM2)
# =========================
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1,3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1,3,1,1)

def normalize_img_0_1(img_bchw):
    return (img_bchw - IMAGENET_MEAN.to(img_bchw.device)) / IMAGENET_STD.to(img_bchw.device)

In [5]:
# =========================
# Cell 4 — Dataset: sequence mode + collate padding + valid mask
# =========================
def bbox_from_mask(mask_hw, pad=5):
    ys, xs = np.where(mask_hw > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(mask_hw.shape[1]-1, x1 + pad); y1 = min(mask_hw.shape[0]-1, y1 + pad)
    return [float(x0), float(y0), float(x1), float(y1)]

def resize_u8_gray(u8_hw, out_hw=512):
    t = torch.from_numpy(u8_hw).unsqueeze(0).unsqueeze(0).float()
    t = F.interpolate(t, size=(out_hw, out_hw), mode="bilinear", align_corners=False)
    return t.squeeze(0).squeeze(0).clamp(0,255).byte().cpu().numpy()

def resize_mask_u8(mask_hw, out_hw=512):
    t = torch.from_numpy(mask_hw.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(out_hw, out_hw), mode="nearest")
    return (t.squeeze(0).squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)

def load_2d_gray_u8(path):
    im = Image.open(path).convert("L")
    return np.array(im, dtype=np.uint8)

def load_2d_mask01(path):
    m = Image.open(path).convert("L")
    return (np.array(m) > 0).astype(np.uint8)

def load_vol_img_msk(path_img, path_msk):
    ext = _ext(path_img)
    if ext in [".nii", ".nii.gz"]:
        img = sitk.GetArrayFromImage(sitk.ReadImage(path_img))
        msk = sitk.GetArrayFromImage(sitk.ReadImage(path_msk))
        return img, msk
    if ext == ".npy":
        return np.load(path_img), np.load(path_msk)
    raise ValueError(f"Unsupported vol ext: {ext} ({path_img})")

def _slice_window(center: int, depth: int, T: int):
    k = T // 2
    start = max(0, center - k)
    end   = min(depth, center + k + 1)
    idxs  = list(range(start, end))
    while len(idxs) < T:
        if idxs[0] > 0:
            idxs = [idxs[0] - 1] + idxs
        elif idxs[-1] < depth - 1:
            idxs = idxs + [idxs[-1] + 1]
        else:
            idxs = [idxs[0]] + idxs
    return idxs[:T]

SEQ_LEN_3D = 5

class Mixed2DFromPairsSeq(Dataset):
    def __init__(self, pairs, target_hw=512, pad=5, seq_len_3d=5):
        self.pairs = pairs
        self.target_hw = target_hw
        self.pad = pad
        self.seq_len_3d = int(seq_len_3d)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # accept (index, task_id) if sampler yields that
        task_id = None
        if isinstance(idx, (tuple, list)) and len(idx) == 2:
            idx, task_id = idx

        rec = self.pairs[int(idx)]
        img_path = rec["image"]
        msk_path = rec["mask"]

        if (not os.path.exists(img_path)) or (not os.path.exists(msk_path)):
            return None

        e = _ext(img_path)

        if e in IMG_EXT_2D:
            img_u8 = load_2d_gray_u8(img_path)
            msk_u8 = load_2d_mask01(msk_path)

            img_r = resize_u8_gray(img_u8, self.target_hw)
            msk_r = resize_mask_u8(msk_u8, self.target_hw)

            box = bbox_from_mask(msk_r, pad=self.pad)
            if box is None:
                return None

            img_rgb = np.repeat(img_r[..., None], 3, axis=2)
            img_t = torch.from_numpy(img_rgb).permute(2,0,1).float() / 255.0
            msk_t = torch.from_numpy(msk_r).unsqueeze(0).float()
            box_t = torch.tensor(box, dtype=torch.float32)

            imgs = img_t.unsqueeze(0)  # [1,3,H,W]
            msks = msk_t.unsqueeze(0)  # [1,1,H,W]
            boxs = box_t.unsqueeze(0)  # [1,4]

        else:
            vol, msk = load_vol_img_msk(img_path, msk_path)
            msk_bin = (msk > 0).astype(np.uint8)

            areas = msk_bin.reshape(msk_bin.shape[0], -1).sum(1)
            if areas.max() == 0:
                return None
            center = int(np.argmax(areas))
            idxs = _slice_window(center, msk_bin.shape[0], self.seq_len_3d)

            imgs_list, msks_list, box_list = [], [], []
            for z in idxs:
                sl = vol[z].astype(np.float32)
                lo, hi = np.percentile(sl, 0.5), np.percentile(sl, 99.5)
                if hi <= lo:
                    hi = lo + 1.0
                sl = np.clip(sl, lo, hi)
                sl = (255.0 * (sl - lo) / (hi - lo)).clip(0,255).astype(np.uint8)

                img_r = resize_u8_gray(sl, self.target_hw)
                msk_r = resize_mask_u8(msk_bin[z], self.target_hw)

                box = bbox_from_mask(msk_r, pad=self.pad)
                if box is None:
                    box = [0.0, 0.0, float(self.target_hw-1), float(self.target_hw-1)]

                img_rgb = np.repeat(img_r[..., None], 3, axis=2)
                img_t = torch.from_numpy(img_rgb).permute(2,0,1).float() / 255.0
                msk_t = torch.from_numpy(msk_r).unsqueeze(0).float()
                box_t = torch.tensor(box, dtype=torch.float32)

                imgs_list.append(img_t)
                msks_list.append(msk_t)
                box_list.append(box_t)

            imgs = torch.stack(imgs_list, 0)  # [T,3,H,W]
            msks = torch.stack(msks_list, 0)  # [T,1,H,W]
            boxs = torch.stack(box_list, 0)   # [T,4]

        meta = {
            "dataset": rec.get("dataset"),
            "subdataset": rec.get("subdataset"),
            "modality": rec.get("modality"),
            "pipeline": rec.get("pipeline"),
            "identifier": rec.get("identifier"),
            "class": rec.get("class"),
            "dim": rec.get("dim"),
            "image": img_path,
            "mask": msk_path,
            "pair_index": int(idx),
            "task_id": task_id,
        }
        return imgs, msks, boxs, meta

def collate_pad_none_seq(batch):
    """
    Returns:
      imgs:   [B, T, 3, H, W]
      msks:   [B, T, 1, H, W]
      boxes:  [B, T, 4]
      valid:  [B, T] bool
      metas:  list[dict]
    """
    batch = [b for b in batch if b is not None]
    if not batch:
        return None

    imgs_list, msks_list, boxs_list, metas = zip(*batch)
    T_list = [x.shape[0] for x in imgs_list]
    T_max = max(T_list)

    imgs_pad, msks_pad, boxs_pad, valids = [], [], [], []
    for imgs, msks, boxs, T in zip(imgs_list, msks_list, boxs_list, T_list):
        if T < T_max:
            pad_n = T_max - T
            imgs = torch.cat([imgs, imgs[-1:].repeat(pad_n, 1, 1, 1)], dim=0)
            msks = torch.cat([msks, msks[-1:].repeat(pad_n, 1, 1, 1)], dim=0)
            boxs = torch.cat([boxs, boxs[-1:].repeat(pad_n, 1)], dim=0)

        valid = torch.zeros(T_max, dtype=torch.bool)
        valid[:T] = True

        imgs_pad.append(imgs)
        msks_pad.append(msks)
        boxs_pad.append(boxs)
        valids.append(valid)

    return (
        torch.stack(imgs_pad, 0),
        torch.stack(msks_pad, 0),
        torch.stack(boxs_pad, 0),
        torch.stack(valids, 0),
        list(metas),
    )

In [6]:
# ---- SAFE loader for BalancedTaskBatchSampler (bypasses rwkv_medsam2/__init__.py -> ext.vrwkv)
import sys, importlib.util, importlib.machinery

RWKV_ROOT_CANDIDATES = [
    "/data/jrbonill/RWKV-MedSAM2",
    "/data/jrbonill/rwkv_medsam2",
    "/RWKV-MedSAM2",
]

def _find_rwkv_dataset_py():
    for root in RWKV_ROOT_CANDIDATES:
        ds_path = os.path.join(root, "rwkv_medsam2", "dataset.py")
        if os.path.isfile(ds_path):
            return ds_path
    return None

def load_balanced_task_batch_sampler():
    try:
        from rwkv_medsam2.dataset import BalancedTaskBatchSampler
        return BalancedTaskBatchSampler
    except Exception as e:
        ds_path = _find_rwkv_dataset_py()
        if ds_path is None:
            raise RuntimeError(
                "Could not find rwkv_medsam2/dataset.py in RWKV_ROOT_CANDIDATES. "
                "Update RWKV_ROOT_CANDIDATES to your RWKV-MedSAM2 repo path."
            ) from e

        name = "rwkv_medsam2_dataset_standalone"
        loader = importlib.machinery.SourceFileLoader(name, ds_path)
        spec = importlib.util.spec_from_loader(loader.name, loader)
        mod = importlib.util.module_from_spec(spec)
        loader.exec_module(mod)

        if not hasattr(mod, "BalancedTaskBatchSampler"):
            raise RuntimeError(f"BalancedTaskBatchSampler not found in: {ds_path}")
        return mod.BalancedTaskBatchSampler

BalancedTaskBatchSampler = load_balanced_task_batch_sampler()
print("Loaded BalancedTaskBatchSampler:", BalancedTaskBatchSampler)

/usr/local/lib/python3.10/dist-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.10/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.10/dist-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.10/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


Loaded BalancedTaskBatchSampler: <class 'rwkv_medsam2_dataset_standalone.BalancedTaskBatchSampler'>


In [7]:
# =========================
# Cell 5 — Loaders: BalancedTaskBatchSampler + fixed batches
# (Uses SAFE-loaded BalancedTaskBatchSampler from Cell 0)
# =========================
train_ds = Mixed2DFromPairsSeq(train_pairs_small, target_hw=TARGET_HW, pad=5, seq_len_3d=SEQ_LEN_3D)
val_ds   = Mixed2DFromPairsSeq(val_pairs_small,   target_hw=TARGET_HW, pad=5, seq_len_3d=SEQ_LEN_3D)

def pairs_to_sampler_sequences(pairs_list):
    seqs = []
    for rec in pairs_list:
        task = rec.get("class") or f"{rec.get('dataset','unknown')}/{rec.get('subdataset','default')}"
        seqs.append({
            "dataset": rec.get("dataset","unknown"),
            "subdataset": rec.get("subdataset","default"),
            "tasks": [str(task)],
            "dim": int(rec.get("dim", 2)),
            "modality": rec.get("modality","unknown"),
            "sequence": [(rec["image"], rec["mask"])],
        })
    return seqs

def build_tasks_map_from_pairs(pairs_list):
    tasks_map = {}
    for rec in pairs_list:
        ds = rec.get("dataset","unknown")
        sd = rec.get("subdataset","default")
        task = rec.get("class") or f"{ds}/{sd}"
        task = str(task)
        if task not in tasks_map:
            tasks_map[task] = {"classes": [], "datasets": {}}
        tasks_map[task]["datasets"].setdefault(ds, set()).add(sd)
    for t in list(tasks_map.keys()):
        tasks_map[t]["datasets"] = {k: sorted(list(v)) for k, v in tasks_map[t]["datasets"].items()}
    return tasks_map

# IMPORTANT: do NOT import rwkv_medsam2.dataset here.
# BalancedTaskBatchSampler must come from the SAFE loader in Cell 0.
sampler_train_seqs = pairs_to_sampler_sequences(train_pairs_small)
tasks_map = build_tasks_map_from_pairs(train_pairs_small)

BATCH_SIZE = 4
MAX_TRAIN_BATCHES = 200

base_batch_sampler = BalancedTaskBatchSampler(
    sequences=sampler_train_seqs,
    tasks_map=tasks_map,
    batch_size=BATCH_SIZE,
    drop_last=True,
    seed=SEED,
    num_samples=MAX_TRAIN_BATCHES * BATCH_SIZE,
)

class FixedBatchListSampler:
    def __init__(self, batch_sampler, max_batches: int):
        self.batches = []
        for i, b in enumerate(batch_sampler):
            self.batches.append(b)
            if i + 1 >= max_batches:
                break
    def __iter__(self):
        yield from self.batches
    def __len__(self):
        return len(self.batches)

fixed_train_batch_sampler = FixedBatchListSampler(base_batch_sampler, MAX_TRAIN_BATCHES)

train_loader = DataLoader(
    train_ds,
    batch_sampler=fixed_train_batch_sampler,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=collate_pad_none_seq,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_pad_none_seq,
)

b = next(iter(train_loader))
imgs, msks, boxes, valid, metas = b
print("imgs:", imgs.shape, imgs.dtype, float(imgs.min()), float(imgs.max()))
print("msks:", msks.shape, msks.dtype, float(msks.min()), float(msks.max()))
print("boxes:", boxes.shape, boxes.dtype)
print("valid:", valid.shape, valid.dtype, "valid_count:", int(valid.sum()))
print("meta sample:", metas[0])
print("Fixed train batches:", len(fixed_train_batch_sampler))

imgs: torch.Size([4, 1, 3, 512, 512]) torch.float32 0.0 1.0
msks: torch.Size([4, 1, 1, 512, 512]) torch.float32 0.0 1.0
boxes: torch.Size([4, 1, 4]) torch.float32
valid: torch.Size([4, 1]) torch.bool valid_count: 4
meta sample: {'dataset': 'ThyroidUltra', 'subdataset': 'default', 'modality': 'ultrasound', 'pipeline': '2D', 'identifier': 'train_16968', 'class': 'thyroid_nodule', 'dim': 2, 'image': '/data/Preprocessed/ThyroidUltra/ultrasound/default/train/16968/images/cc130016_img000.png', 'mask': '/data/Preprocessed/ThyroidUltra/ultrasound/default/train/16968/masks/cc130016_img000_mask000_%thyroid_nodule%_comp000.png', 'pair_index': 36805, 'task_id': 'thyroid_nodule'}
Fixed train batches: 141


In [8]:
# =========================
# Cell 6 — MedSAM2 model setup
# =========================
MEDSAM2_BASE = "/data/jrbonill/medsam2_env/MedSAM2"
CFG_NAME     = "sam2.1_hiera_t512.yaml"
CKPT_PATH    = f"{MEDSAM2_BASE}/checkpoints/MedSAM2_latest.pt"

assert os.path.isfile(CKPT_PATH), f"Missing ckpt: {CKPT_PATH}"

import sys
if MEDSAM2_BASE not in sys.path:
    sys.path.append(MEDSAM2_BASE)

from hydra import initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

CONFIG_DIR = f"{MEDSAM2_BASE}/sam2/configs"
assert os.path.isdir(CONFIG_DIR), f"Missing config dir: {CONFIG_DIR}"

GlobalHydra.instance().clear()
initialize_config_dir(config_dir=CONFIG_DIR, version_base=None)

from sam2.build_sam import build_sam2

model = build_sam2(CFG_NAME)
ckpt = torch.load(CKPT_PATH, map_location="cpu")
missing, unexpected = model.load_state_dict(ckpt, strict=False)
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

model = model.to(DEVICE).train()

Missing keys: 471
Unexpected keys: 1


In [9]:
# =========================
# Cell 7 — Forward functions (MedSAM2 box prompt + sequence wrapper)
# =========================
def forward_medsam2_box(model, imgs_b3hw_01, boxes_b4):
    """
    imgs_b3hw_01: [B,3,512,512] float in [0,1]
    boxes_b4: [B,4] (x0,y0,x1,y1) in pixel coords on 512x512
    returns: logits [B,1,512,512]
    """
    B = imgs_b3hw_01.shape[0]

    # REQUIRED: MedSAM2 ImageNet normalization
    x = normalize_img_0_1(imgs_b3hw_01)

    backbone_out = model.forward_image(x)
    _, vision_feats, vision_pos_embeds, feat_sizes = model._prepare_backbone_features(backbone_out)

    feats = [
        feat.permute(1, 2, 0).reshape(B, -1, *sz)
        for feat, sz in zip(vision_feats[::-1], feat_sizes[::-1])
    ][::-1]
    image_embed = feats[-1]
    hires_feats = feats[:-1]

    boxes = boxes_b4.view(B, 2, 2)

    sparse, dense = model.sam_prompt_encoder(points=None, boxes=boxes, masks=None)

    dense = F.interpolate(dense, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)

    image_pe = model.sam_prompt_encoder.get_dense_pe()
    if image_pe.shape[-2:] != image_embed.shape[-2:]:
        image_pe = F.interpolate(image_pe, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)
    image_pe = image_pe.to(image_embed.dtype)

    lowres_logits, _, _, _ = model.sam_mask_decoder(
        image_embeddings=image_embed,
        image_pe=image_pe,
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        multimask_output=False,
        repeat_image=False,
        high_res_features=hires_feats,
    )

    logits = F.interpolate(lowres_logits, size=(TARGET_HW, TARGET_HW), mode="bilinear", align_corners=False)
    return logits

def forward_medsam2_box_sequence(model, imgs_bt3hw_01, boxes_bt4):
    """
    imgs_bt3hw_01: [B,T,3,512,512]
    boxes_bt4:     [B,T,4]
    returns logits [B,T,1,512,512]
    """
    B, T = imgs_bt3hw_01.shape[:2]
    flat_imgs  = imgs_bt3hw_01.reshape(B*T, 3, TARGET_HW, TARGET_HW)
    flat_boxes = boxes_bt4.reshape(B*T, 4)
    flat_logits = forward_medsam2_box(model, flat_imgs, flat_boxes)
    return flat_logits.reshape(B, T, 1, TARGET_HW, TARGET_HW)

In [10]:
# =========================
# Cell 8 — TensorBoard setup (Oxford-style)
# =========================
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

%load_ext tensorboard

EPOCHS = 20
MAX_VAL_BATCHES = 50
LOG_EVERY_STEPS = 25
PRINT_VAL_EVERY_EPOCH = True

lr = 1e-5
wd = 0.01
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

tb_dir = os.path.join(OUT_ROOT, "tb")
run_dir = os.path.join(
    tb_dir,
    datetime.now().strftime("%Y%m%d-%H%M%S") + f"-seed{SEED}-bs{BATCH_SIZE}"
)

writer = SummaryWriter(log_dir=run_dir, flush_secs=5, max_queue=10)

# Match Oxford's add_hparams pattern exactly
writer.add_hparams(
    {
        "bs": int(BATCH_SIZE),
        "epochs": int(EPOCHS),
        "lr": float(lr),
        "weight_decay": float(wd),
        "seed": int(SEED),
        "max_train_batches": int(MAX_TRAIN_BATCHES),
        "max_val_batches": int(MAX_VAL_BATCHES),
    },
    {"val/iou_best": 0.0}
)

print("TensorBoard run_dir:", run_dir)

TensorBoard run_dir: /data/jrbonill/medsam2_refine_runs/medsam2_refine_20260128_173900/tb/20260128-173926-seed42-bs4


In [11]:
# =========================
# Cell 9 (optional) — Launch TensorBoard inline
# =========================
import subprocess, time

PORT = 6007
HOST = "127.0.0.1"

subprocess.run(["bash", "-lc", f"pkill -f 'tensorboard.*--port {PORT}' || true"], check=False)

proc = subprocess.Popen(
    ["tensorboard", "--logdir", run_dir, "--port", str(PORT), "--host", HOST, "--reload_multifile", "true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("TensorBoard PID:", proc.pid)
print(f"Now serving on http://{HOST}:{PORT}")
time.sleep(1)

TensorBoard PID: 145884
Now serving on http://127.0.0.1:6007


In [12]:
# =========================
# Cell 10 (optional) — Display TensorBoard
# =========================
from IPython.display import IFrame, display
display(IFrame(src="http://127.0.0.1:6007", width="100%", height=900))

In [13]:
# =========================
# Cell 11 — Train/Val functions + main loop (Oxford-style, WITH per-task + per-modality TB summaries)
# Alphabetical ordering for per-task and per-modality tables.
# =========================
from tqdm.auto import tqdm

def get_lr_group(opt, group_idx=0):
    if not hasattr(opt, "param_groups") or len(opt.param_groups) == 0:
        return float("nan")
    group_idx = min(max(0, group_idx), len(opt.param_groups) - 1)
    return float(opt.param_groups[group_idx].get("lr", float("nan")))

def bce_dice_loss(logits, gt):
    bce = F.binary_cross_entropy_with_logits(logits, gt, reduction="mean")
    probs = torch.sigmoid(logits)
    inter = (probs * gt).flatten(1).sum(1)
    denom = probs.flatten(1).sum(1) + gt.flatten(1).sum(1) + 1e-6
    dice = 1.0 - (2 * inter / denom).mean()
    return bce + dice

@torch.no_grad()
def dice_torch_from_logits(logits_b1hw, gt_b1hw):
    probs = torch.sigmoid(logits_b1hw)
    pred = (probs > 0.5).float()
    inter = (pred * gt_b1hw).flatten(1).sum(1)
    denom = pred.flatten(1).sum(1) + gt_b1hw.flatten(1).sum(1) + 1e-6
    return (2 * inter / denom).mean()

@torch.no_grad()
def iou_torch_from_logits(logits_b1hw, gt_b1hw, thr=0.5):
    probs = torch.sigmoid(logits_b1hw)
    pred = (probs > thr).float()
    inter = (pred * gt_b1hw).flatten(1).sum(1)
    union = (pred + gt_b1hw - pred * gt_b1hw).flatten(1).sum(1) + 1e-6
    return (inter / union).mean()

@torch.no_grad()
def dice_iou_best_from_logits(
    logits_n1hw: torch.Tensor,
    gt_n1hw: torch.Tensor,
    thr_vals=(0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9),
):
    device = logits_n1hw.device
    probs  = torch.sigmoid(logits_n1hw).to(torch.float32)  # [N,1,H,W]
    gt     = (gt_n1hw > 0.5).to(torch.float32)             # [N,1,H,W]

    p = probs.flatten(2)  # [N,1,HW]
    g = gt.flatten(2)     # [N,1,HW]

    thr = torch.tensor(thr_vals, device=device, dtype=p.dtype)  # [K]
    K = thr.numel()
    TH = thr.view(K, 1, 1, 1)

    pred = (p.unsqueeze(0) >= TH)                 # [K,N,1,HW]
    g_bool = (g > 0.5).to(torch.bool).unsqueeze(0)

    inter = (pred & g_bool).sum(dim=-1).to(torch.float32)       # [K,N,1]
    p_sum = pred.sum(dim=-1).to(torch.float32)                  # [K,N,1]
    g_sum = g_bool.sum(dim=-1).to(torch.float32).expand_as(p_sum)

    dice = (2.0 * inter) / (p_sum + g_sum).clamp_min(1.0)       # [K,N,1]
    union = (p_sum + g_sum - inter).clamp_min(1.0)              # [K,N,1]
    iou  = inter / union                                        # [K,N,1]

    dice_mean = dice.mean(dim=1).squeeze(-1)  # [K]
    iou_mean  = iou.mean(dim=1).squeeze(-1)   # [K]

    try:
        k05 = int((thr == 0.5).nonzero(as_tuple=True)[0].item())
    except Exception:
        k05 = int(torch.argmin((thr - 0.5).abs()).item())

    best_k   = int(torch.argmax(dice_mean).item())
    best_thr = float(thr[best_k].item())

    return (
        float(dice_mean[k05].item()),
        float(iou_mean[k05].item()),
        float(dice_mean[best_k].item()),
        float(iou_mean[best_k].item()),
        best_thr,
    )

def _select_valid_frames(flat_logits, flat_msks, flat_valid_bool):
    if flat_valid_bool is None:
        return flat_logits, flat_msks
    flat_valid_bool = flat_valid_bool.to(device=flat_logits.device)
    idx = torch.nonzero(flat_valid_bool, as_tuple=False).squeeze(1)
    if idx.numel() == 0:
        return None, None
    return flat_logits.index_select(0, idx), flat_msks.index_select(0, idx)

def _task_key_from_meta(meta: dict) -> str:
    t = meta.get("class", None)
    if t is None or str(t).strip() == "":
        ds = meta.get("dataset", "unknown")
        sd = meta.get("subdataset", "default")
        return f"{ds}/{sd}"
    return str(t)

def _mod_key_from_meta(meta: dict) -> str:
    m = meta.get("modality", None)
    return str(m).strip().lower() if m is not None else "unknown"

def _sanitize_tb_key(s: str) -> str:
    s = str(s)
    s = s.replace(" ", "_").replace("/", "_").replace("\\", "_").replace(":", "_").replace("|", "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s[:200]

def _group_stats_init():
    return {"dice@0.5": [], "iou@0.5": [], "dice_best": [], "iou_best": [], "best_thr": [], "n": 0}

def _group_stats_update(stats_map: dict, key: str, d05: float, j05: float, db: float, jb: float, thr: float):
    if key not in stats_map:
        stats_map[key] = _group_stats_init()
    stats_map[key]["dice@0.5"].append(float(d05))
    stats_map[key]["iou@0.5"].append(float(j05))
    stats_map[key]["dice_best"].append(float(db))
    stats_map[key]["iou_best"].append(float(jb))
    stats_map[key]["best_thr"].append(float(thr))
    stats_map[key]["n"] += 1

def _group_stats_mean(stats_map: dict):
    out = {}
    for k, v in stats_map.items():
        out[k] = {
            "dice@0.5":  float(np.mean(v["dice@0.5"]))  if v["dice@0.5"]  else 0.0,
            "iou@0.5":   float(np.mean(v["iou@0.5"]))   if v["iou@0.5"]   else 0.0,
            "dice_best": float(np.mean(v["dice_best"])) if v["dice_best"] else 0.0,
            "iou_best":  float(np.mean(v["iou_best"]))  if v["iou_best"]  else 0.0,
            "best_thr":  float(np.mean(v["best_thr"]))  if v["best_thr"]  else 0.5,
            "n":         int(v["n"]),
        }
    return out

def _md_table_from_group_means(group_means: dict, title: str, alphabetical=True):
    """
    Render a markdown table with fixed columns (Oxford-style "columns").
    Alphabetical ordering by group name (case-insensitive) when alphabetical=True.
    """
    items = list(group_means.items())
    if alphabetical:
        items.sort(key=lambda kv: str(kv[0]).lower())
    else:
        items.sort(key=lambda kv: (kv[1].get("n", 0), kv[1].get("iou_best", 0.0)), reverse=True)

    lines = []
    lines.append(f"**{title}**")
    lines.append("")
    lines.append("| group | n | dice@0.5 | iou@0.5 | dice_best | iou_best | best_thr |")
    lines.append("|---|---:|---:|---:|---:|---:|---:|")

    if not items:
        lines.append("| *(none)* | 0 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.50 |")
        return "\n".join(lines)

    for name, m in items:
        lines.append(
            f"| {str(name)} | {int(m['n'])} | {m['dice@0.5']:.4f} | {m['iou@0.5']:.4f} | "
            f"{m['dice_best']:.4f} | {m['iou_best']:.4f} | {m['best_thr']:.2f} |"
        )

    return "\n".join(lines)

@torch.no_grad()
def _per_sample_metrics_from_batch(logits_bt1hw, msks_bt1hw, valid_bt):
    B, T = logits_bt1hw.shape[:2]
    out = []
    for i in range(B):
        v = valid_bt[i].reshape(-1)
        idx = torch.nonzero(v, as_tuple=False).squeeze(1)
        if idx.numel() == 0:
            out.append((0.0, 0.0, 0.0, 0.0, 0.5, 0))
            continue
        li = logits_bt1hw[i].index_select(0, idx)  # [Ti,1,H,W]
        mi = msks_bt1hw[i].index_select(0, idx)    # [Ti,1,H,W]
        d05, j05, db, jb, thrb = dice_iou_best_from_logits(li, mi)
        out.append((d05, j05, db, jb, thrb, int(idx.numel())))
    return out

@torch.no_grad()
def validate_epoch_like_train_sam2(model, loader, epoch, writer, max_batches=None):
    model.eval()

    dices_05, ious_05, dices_best, ious_best, best_thrs = [], [], [], [], []
    task_stats, mod_stats = {}, {}

    n_frames, n_samples = 0, 0

    pbar = tqdm(loader, desc=f"Val Epoch {epoch}", unit="batch", leave=False)
    for bi, batch in enumerate(pbar, 1):
        if batch is None:
            continue

        imgs, msks, boxes, valid, metas = batch
        imgs  = imgs.to(DEVICE, non_blocking=True)
        msks  = msks.to(DEVICE, non_blocking=True)
        boxes = boxes.to(DEVICE, non_blocking=True)
        valid = valid.to(DEVICE, non_blocking=True)

        logits_bt1hw = forward_medsam2_box_sequence(model, imgs, boxes)

        flat_logits = logits_bt1hw.reshape(-1, 1, TARGET_HW, TARGET_HW)
        flat_msks   = msks.reshape(-1, 1, TARGET_HW, TARGET_HW)
        flat_valid  = valid.reshape(-1)
        flat_logits, flat_msks = _select_valid_frames(flat_logits, flat_msks, flat_valid)
        if flat_logits is None:
            continue

        d05, j05, db, jb, thrb = dice_iou_best_from_logits(flat_logits, flat_msks)

        dices_05.append(float(d05))
        ious_05.append(float(j05))
        dices_best.append(float(db))
        ious_best.append(float(jb))
        best_thrs.append(float(thrb))

        per_s = _per_sample_metrics_from_batch(logits_bt1hw, msks, valid)
        for meta_i, (sd05, sj05, sdb, sjb, sth, nvi) in zip(metas, per_s):
            if nvi <= 0:
                continue
            _group_stats_update(task_stats, _task_key_from_meta(meta_i), sd05, sj05, sdb, sjb, sth)
            _group_stats_update(mod_stats,  _mod_key_from_meta(meta_i),  sd05, sj05, sdb, sjb, sth)

        n_frames  += int(flat_msks.shape[0])
        n_samples += int(imgs.shape[0])

        pbar.set_postfix(
            dice=f"{dices_05[-1]:.4f}",
            iou=f"{ious_05[-1]:.4f}",
            dice_best=f"{dices_best[-1]:.4f}",
            iou_best=f"{ious_best[-1]:.4f}",
            thr=f"{best_thrs[-1]:.2f}",
            frames=n_frames,
        )

        if (max_batches is not None) and (bi >= max_batches):
            break

    metrics = {
        "dice@0.5":  float(np.mean(dices_05))   if dices_05   else 0.0,
        "iou@0.5":   float(np.mean(ious_05))    if ious_05    else 0.0,
        "dice_best": float(np.mean(dices_best)) if dices_best else 0.0,
        "iou_best":  float(np.mean(ious_best))  if ious_best  else 0.0,
        "best_thr":  float(np.mean(best_thrs))  if best_thrs  else 0.5,
        "n_frames":  int(n_frames),
        "n_batches": int(len(dices_05)),
        "n_samples": int(n_samples),
    }

    for k, v in metrics.items():
        writer.add_scalar(f"val/{k}", float(v), epoch)

    task_means = _group_stats_mean(task_stats)
    mod_means  = _group_stats_mean(mod_stats)

    for task, m in task_means.items():
        tkey = _sanitize_tb_key(task)
        writer.add_scalar(f"val_task/{tkey}/dice@0.5",  float(m["dice@0.5"]),  epoch)
        writer.add_scalar(f"val_task/{tkey}/iou@0.5",   float(m["iou@0.5"]),   epoch)
        writer.add_scalar(f"val_task/{tkey}/dice_best", float(m["dice_best"]), epoch)
        writer.add_scalar(f"val_task/{tkey}/iou_best",  float(m["iou_best"]),  epoch)
        writer.add_scalar(f"val_task/{tkey}/best_thr",  float(m["best_thr"]),  epoch)
        writer.add_scalar(f"val_task/{tkey}/n",         float(m["n"]),         epoch)

    for mod, m in mod_means.items():
        mkey = _sanitize_tb_key(mod)
        writer.add_scalar(f"val_modality/{mkey}/dice@0.5",  float(m["dice@0.5"]),  epoch)
        writer.add_scalar(f"val_modality/{mkey}/iou@0.5",   float(m["iou@0.5"]),   epoch)
        writer.add_scalar(f"val_modality/{mkey}/dice_best", float(m["dice_best"]), epoch)
        writer.add_scalar(f"val_modality/{mkey}/iou_best",  float(m["iou_best"]),  epoch)
        writer.add_scalar(f"val_modality/{mkey}/best_thr",  float(m["best_thr"]),  epoch)
        writer.add_scalar(f"val_modality/{mkey}/n",         float(m["n"]),         epoch)

    # Oxford-style: column tables (Markdown), alphabetically ordered
    overall_table = "\n".join([
        f"**[Epoch {epoch}] Validation (overall)**",
        "",
        "| group | n | dice@0.5 | iou@0.5 | dice_best | iou_best | best_thr |",
        "|---|---:|---:|---:|---:|---:|---:|",
        f"| overall | {int(metrics['n_samples'])} | {metrics['dice@0.5']:.4f} | {metrics['iou@0.5']:.4f} | {metrics['dice_best']:.4f} | {metrics['iou_best']:.4f} | {metrics['best_thr']:.2f} |",
    ])

    task_table = _md_table_from_group_means(
        task_means, title=f"[Epoch {epoch}] Validation per-task", alphabetical=True
    )
    mod_table = _md_table_from_group_means(
        mod_means,  title=f"[Epoch {epoch}] Validation per-modality", alphabetical=True
    )

    writer.add_text("val/summary", overall_table, epoch)
    writer.add_text("val/task_summary", task_table, epoch)
    writer.add_text("val/modality_summary", mod_table, epoch)

    writer.flush()
    model.train()
    return metrics

def train_epoch_like_train_sam2(model, train_loader, optimizer, scaler, epoch, writer, global_step, LOG_EVERY_STEPS=25):
    model.train()

    ema = {"data": None, "step": None}
    def ema_update(key, val, alpha=0.1):
        if ema[key] is None:
            ema[key] = float(val)
        else:
            ema[key] = (1 - alpha) * ema[key] + alpha * float(val)

    losses, dices = [], []
    task_stats, mod_stats = {}, {}

    pbar = tqdm(train_loader, desc=f"Train Epoch {epoch}", unit="batch", leave=True)

    end_prev = time.perf_counter()
    for step, batch in enumerate(pbar, 1):
        data_wait = time.perf_counter() - end_prev
        if batch is None:
            end_prev = time.perf_counter()
            continue

        imgs, msks, boxes, valid, metas = batch
        imgs  = imgs.to(DEVICE, non_blocking=True)
        msks  = msks.to(DEVICE, non_blocking=True)
        boxes = boxes.to(DEVICE, non_blocking=True)
        valid = valid.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits_bt1hw = forward_medsam2_box_sequence(model, imgs, boxes)

            flat_logits  = logits_bt1hw.reshape(-1, 1, TARGET_HW, TARGET_HW)
            flat_msks    = msks.reshape(-1, 1, TARGET_HW, TARGET_HW)
            flat_valid   = valid.reshape(-1)

            flat_logits, flat_msks = _select_valid_frames(flat_logits, flat_msks, flat_valid)
            if flat_logits is None:
                end_prev = time.perf_counter()
                continue

            loss = bce_dice_loss(flat_logits, flat_msks)
            dice = dice_torch_from_logits(flat_logits, flat_msks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        step_time = time.perf_counter() - t0

        losses.append(float(loss.detach().cpu().item()))
        dices.append(float(dice.detach().cpu().item()))
        ema_update("data", data_wait)
        ema_update("step", step_time)

        # per-task/per-modality epoch-level stats (from per-sample dice/iou sweep)
        with torch.no_grad():
            per_s = _per_sample_metrics_from_batch(logits_bt1hw.detach(), msks, valid)
            for meta_i, (sd05, sj05, sdb, sjb, sth, nvi) in zip(metas, per_s):
                if nvi <= 0:
                    continue
                _group_stats_update(task_stats, _task_key_from_meta(meta_i), sd05, sj05, sdb, sjb, sth)
                _group_stats_update(mod_stats,  _mod_key_from_meta(meta_i),  sd05, sj05, sdb, sjb, sth)

        pbar.set_postfix(
            loss=f"{losses[-1]:.4f}",
            dice=f"{dices[-1]:.4f}",
            data=f"{ema['data']:.3f}s",
            step=f"{ema['step']:.3f}s",
        )

        if (step % LOG_EVERY_STEPS) == 0:
            writer.add_scalar("train/loss", float(np.mean(losses[-LOG_EVERY_STEPS:])), global_step)
            writer.add_scalar("train/dice@0.5", float(np.mean(dices[-LOG_EVERY_STEPS:])), global_step)
            writer.add_scalar("train/lr", get_lr_group(optimizer, 0), global_step)
            writer.add_scalar("train/time_data_s", float(ema["data"]), global_step)
            writer.add_scalar("train/time_step_s", float(ema["step"]), global_step)

        global_step += 1
        end_prev = time.perf_counter()

    avg_loss = float(np.mean(losses)) if losses else 0.0
    avg_dice = float(np.mean(dices))  if dices  else 0.0

    writer.add_scalar("train_epoch/loss", avg_loss, epoch)
    writer.add_scalar("train_epoch/dice@0.5", avg_dice, epoch)

    task_means = _group_stats_mean(task_stats)
    mod_means  = _group_stats_mean(mod_stats)

    for task, m in task_means.items():
        tkey = _sanitize_tb_key(task)
        writer.add_scalar(f"train_task/{tkey}/dice@0.5",  float(m["dice@0.5"]),  epoch)
        writer.add_scalar(f"train_task/{tkey}/iou@0.5",   float(m["iou@0.5"]),   epoch)
        writer.add_scalar(f"train_task/{tkey}/dice_best", float(m["dice_best"]), epoch)
        writer.add_scalar(f"train_task/{tkey}/iou_best",  float(m["iou_best"]),  epoch)
        writer.add_scalar(f"train_task/{tkey}/best_thr",  float(m["best_thr"]),  epoch)
        writer.add_scalar(f"train_task/{tkey}/n",         float(m["n"]),         epoch)

    for mod, m in mod_means.items():
        mkey = _sanitize_tb_key(mod)
        writer.add_scalar(f"train_modality/{mkey}/dice@0.5",  float(m["dice@0.5"]),  epoch)
        writer.add_scalar(f"train_modality/{mkey}/iou@0.5",   float(m["iou@0.5"]),   epoch)
        writer.add_scalar(f"train_modality/{mkey}/dice_best", float(m["dice_best"]), epoch)
        writer.add_scalar(f"train_modality/{mkey}/iou_best",  float(m["iou_best"]),  epoch)
        writer.add_scalar(f"train_modality/{mkey}/best_thr",  float(m["best_thr"]),  epoch)
        writer.add_scalar(f"train_modality/{mkey}/n",         float(m["n"]),         epoch)

    # Oxford-style: column tables (Markdown), alphabetically ordered
    train_overall_table = "\n".join([
        f"**[Epoch {epoch}] Train (overall)**",
        "",
        "| group | n | dice@0.5 | iou@0.5 | dice_best | iou_best | best_thr |",
        "|---|---:|---:|---:|---:|---:|---:|",
        f"| overall | {int(len(losses))} | {avg_dice:.4f} | {0.0:.4f} | {0.0:.4f} | {0.0:.4f} | {0.50:.2f} |",
    ])

    train_task_table = _md_table_from_group_means(
        task_means, title=f"[Epoch {epoch}] Train per-task", alphabetical=True
    )
    train_mod_table = _md_table_from_group_means(
        mod_means,  title=f"[Epoch {epoch}] Train per-modality", alphabetical=True
    )

    writer.add_text("train/summary", train_overall_table, epoch)
    writer.add_text("train/task_summary", train_task_table, epoch)
    writer.add_text("train/modality_summary", train_mod_table, epoch)

    return avg_loss, avg_dice, global_step

# -------------------------
# Main loop
# -------------------------
global_step = 0

for epoch in range(EPOCHS):
    avg_loss, avg_dice, global_step = train_epoch_like_train_sam2(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch,
        writer=writer,
        global_step=global_step,
        LOG_EVERY_STEPS=LOG_EVERY_STEPS,
    )

    print(f"[Epoch {epoch}] train: loss={avg_loss:.4f} dice@0.5={avg_dice:.4f}")

    val_metrics = validate_epoch_like_train_sam2(
        model=model,
        loader=val_loader,
        epoch=epoch,
        writer=writer,
        max_batches=MAX_VAL_BATCHES,
    )

    if PRINT_VAL_EVERY_EPOCH:
        print(f"[Epoch {epoch}] val: {val_metrics}")

    ckpt_path = os.path.join(CKPT_OUT, f"epoch_{epoch:03d}.pth")
    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "seed": SEED,
            "global_step": global_step,
            "train_epoch": {"loss": avg_loss, "dice@0.5": avg_dice},
            "val_epoch": val_metrics,
        },
        ckpt_path
    )
    print("Saved:", ckpt_path)

writer.flush()
writer.close()
print("TensorBoard run_dir:", run_dir)

Train Epoch 0:   0%|          | 0/141 [00:00<?, ?batch/s]

/data/jrbonill/medsam2_env/MedSAM2/sam2/modeling/position_encoding.py:133: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  coords = coords @ self.positional_encoding_gaussian_matrix
/data/jrbonill/medsam2_env/MedSAM2/sam2/modeling/sam/mask_decoder.py:234: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not determin

[Epoch 0] train: loss=1.2974 dice@0.5=0.0910


Val Epoch 0:   0%|          | 0/3200 [00:00<?, ?batch/s]

/data/jrbonill/medsam2_env/MedSAM2/sam2/modeling/sam/transformer.py:270: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:838.)
  out = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
/data/jrbonill/medsam2_env/MedSAM2/sam2/modeling/sam/transformer.py:270: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:548.)
  out = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
/data/jrbonill/medsam2_env/MedSAM2/sam2/modeling/sam/transformer.py:270: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:840.)
  out = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
/data/jrbonill/medsam2_env/MedSAM2/sam2/modeling/sam/transformer.py:270: UserWarning: Expected query, key and value to all

...

[Epoch 18] val: {'dice@0.5': 0.32318771939724683, 'iou@0.5': 0.27742735993117096, 'dice_best': 0.33891086366027595, 'iou_best': 0.2889426592364907, 'best_thr': 0.3750000026077032, 'n_frames': 322, 'n_batches': 50, 'n_samples': 198}
Saved: /data/jrbonill/medsam2_refine_runs/medsam2_refine_20260128_173900/checkpoints/epoch_018.pth


Train Epoch 19:   0%|          | 0/141 [00:00<?, ?batch/s]

[Epoch 19] train: loss=0.6474 dice@0.5=0.4576


Val Epoch 19:   0%|          | 0/3200 [00:00<?, ?batch/s]

[Epoch 19] val: {'dice@0.5': 0.3557097705826163, 'iou@0.5': 0.31378203004598615, 'dice_best': 0.3698708128556609, 'iou_best': 0.3233129419386387, 'best_thr': 0.38100000403821466, 'n_frames': 322, 'n_batches': 50, 'n_samples': 198}
Saved: /data/jrbonill/medsam2_refine_runs/medsam2_refine_20260128_173900/checkpoints/epoch_019.pth
TensorBoard run_dir: /data/jrbonill/medsam2_refine_runs/medsam2_refine_20260128_173900/tb/20260128-173926-seed42-bs4
